## GenAI Disclosure Statement

Generative AI tools were used as learning aids. Analysis and conclusions are the team's own work.

---

# Notebook 04 — Fairness and Subgroup Disparity Audit
### DNSC 6330 Responsible Machine Learning Capstone | GWU

**Purpose:** Measure outcome disparities across race, sex, ethnicity, and intersectional subgroups.
Primary metric: Adverse Impact Ratio (AIR). Secondary: FPR/FNR gaps, calibration by group.

**Inputs:** `data/processed/test.parquet`, trained GBM model, operating threshold  
**Outputs:** `tables/master_fairness_table.csv`, `tables/intersectional_table.csv`,
`figures/air_by_threshold.png`, `figures/intersectional_heatmap.png`,
`figures/calibration_by_race.png`

**Lecture 03 connection:** AIR, ME, subgroup disparity, intersectional analysis, statistical tests.


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
import joblib
import glob

from src.fairness import (
    build_fairness_table, air_across_thresholds,
    intersectional_table, add_significance_tests
)
from src.robustness import calibration_by_subgroup

SEED = 42
PROC_DIR   = os.path.join(os.getcwd(), "..", "data", "processed")
MODELS_DIR = os.path.join(os.getcwd(), "..", "models")
TABLES_DIR = os.path.join(os.getcwd(), "..", "tables")
FIG_DIR    = os.path.join(os.getcwd(), "..", "figures")


In [ ]:
# Load test set and model
test_df = pd.read_parquet(os.path.join(PROC_DIR, "test.parquet"))

NON_FEATURE_COLS = [
    "y", "action_taken", "state_code",
    "derived_race", "derived_sex", "derived_ethnicity", "race_sex_intersection"
]
feature_cols = [c for c in test_df.columns if c not in NON_FEATURE_COLS]
X_test = test_df[feature_cols]
y_test = test_df["y"].values

gbm_files = sorted(glob.glob(os.path.join(MODELS_DIR, "gbm_v*.pkl")))
gbm_model = joblib.load(gbm_files[-1])
print(f"Loaded: {gbm_files[-1]}")

# Load threshold from metrics table
metrics_df = pd.read_csv(os.path.join(TABLES_DIR, "metrics_table_final.csv"))
gbm_threshold_row = metrics_df[metrics_df["model"].str.contains("GBM")]
GBM_THRESHOLD = float(gbm_threshold_row["threshold"].values[0]) if len(gbm_threshold_row) > 0 else 0.5
print(f"Operating threshold: {GBM_THRESHOLD}")

# Attach predictions to test_df
test_df = test_df.copy()
test_df["y_prob"] = gbm_model.predict_proba(X_test)[:, 1]
test_df["y_pred"] = (test_df["y_prob"] >= GBM_THRESHOLD).astype(int)

print(f"Test set: {len(test_df):,} rows")
print(f"Overall approval rate at threshold: {test_df['y_pred'].mean():.4f}")


## Section 4.1 — Subgroup Definitions and Counts

Document the subgroup population sizes before computing any metrics.
Small groups require sample-size caveats in the interpretation.


In [ ]:
print("Protected attribute value counts in test set:")
for attr in ["derived_race", "derived_sex", "derived_ethnicity"]:
    if attr in test_df.columns:
        counts = test_df[attr].value_counts()
        print(f"\n  {attr}:")
        for val, cnt in counts.items():
            pct = cnt / len(test_df) * 100
            warn = " ← small sample" if cnt < 200 else ""
            print(f"    {str(val)[:40]:<40} {cnt:>8,}  ({pct:.1f}%){warn}")


## Section 4.2 — Master Fairness Table

All fairness metrics at the operating threshold.
**Reference groups:** White (race), Male (sex), Not Hispanic or Latino (ethnicity).
**AIR < 0.80 is highlighted** — this is the 4/5ths rule threshold.
    "Applicant Sex": "applicant_sex",
    "Applicant Age": "applicant_age",
}
reference_groups = {
    "Race": "White",
    "Sex": "Male",
    "Ethnicity": "Not Hispanic or Latino",
    "Applicant Sex": "Male",
    "Applicant Age": "35-44
    "Sex": "derived_sex",
    "Ethnicity": "derived_ethnicity",
}
reference_groups = {
    "Race": "White",
    "Sex": "Male",
    "Ethnicity": "Not Hispanic or Latino",
}

fairness_table = build_fairness_table(
    test_df,
    y_true_col="y",
    y_prob_col="y_prob",
    threshold=GBM_THRESHOLD,
    protected_attrs=protected_attrs,
    reference_groups=reference_groups,
)

# Add statistical significance tests
fairness_table = add_significance_tests(fairness_table)
fairness_table.to_csv(os.path.join(TABLES_DIR, "master_fairness_table.csv"), index=False)

# Display with AIR flag
def highlight_air(row):
    if not row["is_reference"] and pd.notna(row["air"]):
        if row["air"] < 0.80:
            return ["background-color: #ffe0e0"] * len(row)
    return [""] * len(row)

print("Master Fairness Table (test set, threshold={:.3f}):".format(GBM_THRESHOLD))
try:
    display(fairness_table.style.apply(highlight_air, axis=1))
except:
    display(fairness_table)

# Highlight groups below AIR threshold
below_threshold = fairness_table[
    (~fairness_table["is_reference"]) &
    (fairness_table["air"].notna()) &
    (fairness_table["air"] < 0.80)
]
if len(below_threshold) > 0:
    print(f"\n⚠ GROUPS WITH AIR < 0.80 (4/5ths rule):")
    for _, row in below_threshold.iterrows():
        print(f"  {row['attribute']} — {row['group']}: AIR = {row['air']:.4f}")
else:
    print("\n All groups have AIR  0.80 at the operating threshold.")


## Section 4.3 — AIR Across Multiple Thresholds

AIR values change with threshold. We evaluate at three thresholds to show
that the fairness assessment is not threshold-dependent.


In [ ]:
thresholds_to_test = [0.3, GBM_THRESHOLD, 0.7]
air_multi = air_across_thresholds(
    test_df,
    y_true_col="y",
    y_prob_col="y_prob",
    thresholds=thresholds_to_test,
    protected_attrs=protected_attrs,
    reference_groups=reference_groups,
)
air_multi.to_csv(os.path.join(TABLES_DIR, "air_across_thresholds.csv"), index=False)

# Plot AIR by threshold for Race attribute
fig, ax = plt.subplots(figsize=(10, 6))
race_air = air_multi[(air_multi["attribute"] == "Race") & (~air_multi["is_reference"])]

for group in race_air["group"].unique():
    g_data = race_air[race_air["group"] == group]
    ax.plot(g_data["threshold"], g_data["air"], marker="o", label=group, linewidth=2)

ax.axhline(y=0.80, color="red", linestyle="--", lw=2, label="AIR = 0.80 (4/5ths threshold)")
ax.axhline(y=1.00, color="gray", linestyle=":", lw=1, alpha=0.7, label="Parity (AIR = 1.0)")
ax.set_xlabel("Decision Threshold")
ax.set_ylabel("Adverse Impact Ratio (AIR)")
ax.set_title("AIR by Threshold — Race Groups\n(Reference: White)", fontsize=12)
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=9)
ax.grid(alpha=0.3)
ax.set_ylim(0, 1.2)

plt.tight_layout()
air_path = os.path.join(FIG_DIR, "air_by_threshold.png")
plt.savefig(air_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {air_path}")


## Section 4.4 — Error-Rate Gaps: FNR and FPR

In the fair-lending context, **FNR disparity is the primary concern**: if FNR is higher
for a minority group, that group's qualified applicants are more likely to be incorrectly denied.


In [ ]:
error_rate_table = fairness_table[["attribute", "group", "is_reference", "n", "fpr", "fnr"]].copy()
print("Error-Rate Gaps (FPR and FNR by protected group):")
display(error_rate_table)

# Compute FNR gap
for attr in ["Race", "Sex", "Ethnicity"]:
    attr_df = error_rate_table[error_rate_table["attribute"] == attr]
    ref_fnr = attr_df[attr_df["is_reference"] == True]["fnr"].values
    if len(ref_fnr) == 0:
        continue
    ref_fnr = ref_fnr[0]
    print(f"\n  FNR Gap — {attr} (vs. reference FNR = {ref_fnr:.4f}):")
    for _, row in attr_df[~attr_df["is_reference"]].iterrows():
        if pd.notna(row["fnr"]):
            gap = row["fnr"] - ref_fnr
            pct_higher = (gap / ref_fnr * 100) if ref_fnr > 0 else 0
            flag = "⚠" if abs(pct_higher) > 10 else " "
            print(f"  {flag} {str(row['group'])[:40]:<40}: FNR = {row['fnr']:.4f}  "
                  f"(gap = {gap:+.4f}, {pct_higher:+.1f}% vs. reference)")


## Section 4.5 — Intersectional Analysis: Race × Sex

Intersectional analysis examines whether the combination of race and sex produces disparate
outcomes beyond what either attribute alone would suggest.
Cells with n < 30 are suppressed.


In [ ]:
test_df["_y_pred"] = test_df["y_pred"]
int_table = intersectional_table(
    test_df,
    y_pred_col="_y_pred",
    race_col="derived_race",
    sex_col="derived_sex",
    min_n=30,
    reference_race="White",
    reference_sex="Male",
)
int_table.to_csv(os.path.join(TABLES_DIR, "intersectional_table.csv"), index=False)
display(int_table)

# Heatmap
pivot = int_table.pivot(index="race", columns="sex", values="air_vs_white_male")
fig, ax = plt.subplots(figsize=(9, 6))
mask = pivot.isnull()
sns.heatmap(
    pivot.fillna(0),
    mask=mask,
    annot=True,
    fmt=".3f",
    cmap="RdYlGn",
    vmin=0.5,
    vmax=1.2,
    center=1.0,
    linewidths=0.5,
    ax=ax,
    cbar_kws={"label": "AIR vs. White Male"},
)
# Mark suppressed cells
for (i, j), val in np.ndenumerate(mask.values):
    if val:
        ax.text(j + 0.5, i + 0.5, "n<30", ha="center", va="center",
                color="gray", fontsize=9)
ax.set_title("Intersectional Approval Rate AIR\n(Race × Sex vs. White Male reference)", fontsize=12)
ax.set_xlabel("Sex")
ax.set_ylabel("Race")
plt.tight_layout()
heatmap_path = os.path.join(FIG_DIR, "intersectional_heatmap.png")
plt.savefig(heatmap_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {heatmap_path}")


## Section 4.6 — Calibration by Subgroup

A model may be globally well-calibrated but systematically biased in its probability
estimates for a minority group. We measure Expected Calibration Error (ECE) by race.


In [ ]:
cal_subgroup = calibration_by_subgroup(
    test_df, y_true_col="y", y_prob_col="y_prob", group_col="derived_race"
)
print("Calibration by Race (ECE = Expected Calibration Error, lower is better):")
display(cal_subgroup)
cal_subgroup.to_csv(os.path.join(TABLES_DIR, "calibration_by_race.csv"), index=False)

# Calibration curve plot
from sklearn.calibration import calibration_curve
fig, ax = plt.subplots(figsize=(9, 7))
ax.plot([0, 1], [0, 1], "k--", lw=1, label="Perfect calibration")

colors = plt.cm.tab10.colors
for i, race in enumerate(test_df["derived_race"].dropna().unique()):
    sub = test_df[test_df["derived_race"] == race]
    if len(sub) < 50:
        continue
    try:
        frac_pos, mean_pred = calibration_curve(sub["y"], sub["y_prob"], n_bins=10, strategy="uniform")
        ece_row = cal_subgroup[cal_subgroup["group"] == race]
        ece_val = ece_row["ece"].values[0] if len(ece_row) > 0 else float("nan")
        ax.plot(mean_pred, frac_pos, marker="s", label=f"{race[:25]} (ECE={ece_val:.3f})",
                color=colors[i % len(colors)], linewidth=1.5, markersize=4)
    except Exception:
        pass

ax.set_xlabel("Mean Predicted Probability")
ax.set_ylabel("Fraction of Positives")
ax.set_title("Calibration Curves by Race\n(GBM, Test Set)", fontsize=12)
ax.legend(fontsize=8, bbox_to_anchor=(1.05, 1), loc="upper left")
ax.grid(alpha=0.3)
plt.tight_layout()
cal_path = os.path.join(FIG_DIR, "calibration_by_race.png")
plt.savefig(cal_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {cal_path}")


## Section 4.7 — Fairness Audit Findings

In [ ]:
print("=" * 60)
print("FAIRNESS AUDIT FINDINGS — AUDIT SUMMARY")
print("=" * 60)

# Key AIR findings
non_ref = fairness_table[~fairness_table["is_reference"]]
below_080 = non_ref[non_ref["air"] < 0.80]
above_080 = non_ref[non_ref["air"] >= 0.80]

print(f"\n1. AIR Summary at threshold = {GBM_THRESHOLD}:")
print(f"   Groups with AIR >= 0.80: {len(above_080)}")
print(f"   Groups with AIR <  0.80: {len(below_080)}")

if len(below_080) > 0:
    print("\n   Groups below 0.80 (regulatory concern):")
    for _, r in below_080.iterrows():
        print(f"     {r['attribute']} — {r['group']}: AIR = {r['air']:.4f}")
else:
    print("\n    All measured groups have AIR >= 0.80 at the operating threshold.")

# FNR gap finding
race_df = fairness_table[fairness_table["attribute"] == "Race"]
ref_fnr = race_df[race_df["is_reference"]]["fnr"].values
if len(ref_fnr) > 0:
    ref_fnr = ref_fnr[0]
    max_fnr_gap = race_df[~race_df["is_reference"]]["fnr"].max() - ref_fnr
    worst_fnr_group = race_df[~race_df["is_reference"]].loc[
        race_df[~race_df["is_reference"]]["fnr"].idxmax(), "group"
    ]
    print(f"\n2. FNR gap: Largest gap = {max_fnr_gap:.4f} ({worst_fnr_group} vs. White)")

# Intersectional finding
worst_int = int_table.dropna(subset=["air_vs_white_male"]).sort_values("air_vs_white_male").head(1)
if len(worst_int) > 0:
    r = worst_int.iloc[0]
    print(f"\n3. Worst intersectional cell: {r['race']} × {r['sex']}: AIR = {r['air_vs_white_male']:.4f}")

print("\n Notebook 04 complete.")
